# Solow-Swan Fundamental growth

In [1]:
import numpy as np
from scipy.integrate import solve_ivp

1. Solow-Swan Fundamental Growth Model. This classical model tracks the evolution of capital per worker over time, driven by savings, depreciation, and labor growth.$$\frac{dk(t)}{dt} = s f(k(t)) - (\delta + n) k(t)$$Variables: $k(t)$ is the capital per worker, $s$ is the savings rate, $\delta$ is the depreciation rate, and $n$ is the labor force growth rate.Why it fits Neural Operators: The input space consists of initial capital stocks $k(0)$ or varying parameters $(s, \delta, n)$, and the neural operator learns the entire trajectory function $k(t)$ mapping from the parameter space.

In [3]:
def solow_ode(t, k, s, delta, n, alpha, A):
    """
    Defines the continuous Solow-Swan ODE.
    Cobb-Douglas production function: f(k) = A * k^alpha
    """
    f_k = A * (k ** alpha)
    dk_dt = s * f_k - (delta + n) * k
    return dk_dt

def generate_solow_dataset(num_samples=500, T=50, num_steps=100):
    """
    Generates training points for the Solow-Swan Model.
    
    Args:
        num_samples (int): Number of different parameter combinations (M).
        T (float): Time horizon.
        num_steps (int): Number of time points to evaluate.
        
    Returns:
        X_inputs (np.ndarray): Shape (M, 4) - [k(0), s, delta, n]
        Y_trajectories (np.ndarray): Shape (M, num_steps) - k(t)
    """
    t_span = (0.0, T)
    t_eval = np.linspace(0.0, T, num_steps)
    
    # Pre-allocate arrays
    X_inputs = []
    Y_trajectories = []
    
    # Define fixed economic constants (technology parameter and capital elasticity)
    alpha = 0.3
    A = 2.0
    
    # Set a random seed for reproducibility
    np.random.seed(42)
    
    for _ in range(num_samples):
        # 1. Sample parameters randomly from defined uniform ranges
        s = np.random.uniform(0.1, 0.4)      # Savings rate
        delta = np.random.uniform(0.02, 0.08) # Depreciation rate
        n = np.random.uniform(0.01, 0.03)     # Labor growth rate
        k0 = np.random.uniform(0.5, 5.0)      # Initial capital
        
        # 2. Solve the ODE for the sampled parameters
        sol = solve_ivp(
            solow_ode, 
            t_span, 
            [k0], 
            args=(s, delta, n, alpha, A), 
            t_eval=t_eval,
            method='RK45'
        )
        
        # Append inputs and outputs
        X_inputs.append([k0, s, delta, n])
        Y_trajectories.append(sol.y[0])
        
    return np.array(X_inputs), np.array(Y_trajectories)

In [6]:
M = 100 # Generating 100 sample trajectories for demonstration
X, Y = generate_solow_dataset(num_samples=M)    
print(f"Input Features Shape: {X.shape} (Format: [k0, s, delta, n])")
print(f"Target Trajectories Shape: {Y.shape} (Format: [M, num_time_steps])")

Input Features Shape: (100, 4) (Format: [k0, s, delta, n])
Target Trajectories Shape: (100, 100) (Format: [M, num_time_steps])


# Logistic Growth Model

The logistic ODE is defined as:$$\frac{dL(t)}{dt} = a L(t) - b L(t)^2$$Here, $L(t)$ represents the population or labor size, $a$ is the intrinsic growth rate, and $b$ represents the competition or carrying capacity parameter (Jo et al., 2024).

### Parameter Space

<ul>
  <li>Intrinsic growth rate ($a$): $[0.1, 0.6]$</li>
  <li>Competition parameter ($b$): $[0.001, 0.01]$</li>
  <li>Initial population size ($L(0)$): $[5.0, 50.0]$</li>
</ul>

In [13]:
def logistic_ode(t, L, a, b):
    """
    Defines the continuous Logistic Growth ODE.
    dL/dt = a * L - b * L^2
    """
    dL_dt = a * L - b * (L ** 2)
    return dL_dt

def generate_logistic_dataset(num_samples=500, T=50, num_steps=100):
    """
    Generates training points for the Logistic Growth Model.
    
    Args:
        num_samples (int): Number of different parameter combinations (M).
        T (float): Time horizon.
        num_steps (int): Number of time points to evaluate.
        
    Returns:
        X_inputs (np.ndarray): Shape (M, 3) - [L(0), a, b]
        Y_trajectories (np.ndarray): Shape (M, num_steps) - L(t)
    """
    t_span = (0.0, T)
    t_eval = np.linspace(0.0, T, num_steps)
    
    X_inputs = []
    Y_trajectories = []
    
    np.random.seed(42)
    
    for _ in range(num_samples):
        # 1. Sample parameters from defined uniform ranges
        a = np.random.uniform(0.1, 0.6)
        b = np.random.uniform(0.001, 0.01)
        L0 = np.random.uniform(5.0, 50.0)
        
        # 2. Solve the ODE numerically using scipy.integrate.solve_ivp
        sol = solve_ivp(
            logistic_ode, 
            t_span, 
            [L0], 
            args=(a, b), 
            t_eval=t_eval,
            method='RK45'
        )
        
        # 3. Append inputs and outputs
        X_inputs.append([L0, a, b])
        Y_trajectories.append(sol.y[0])
        
    return np.array(X_inputs), np.array(Y_trajectories)

In [23]:
M = 100
X, Y = generate_logistic_dataset(num_samples=M)
    
print(f"Input Features Shape: {X.shape} (Format: [L0, a, b])")
print(f"Target Trajectories Shape: {Y.shape} (Format: [M, num_time_steps])")

Input Features Shape: (100, 3) (Format: [L0, a, b])
Target Trajectories Shape: (100, 100) (Format: [M, num_time_steps])


# The Ramsey-Cass-Koopmans (RCK) Growth System

3. The Ramsey-Cass-Koopmans (RCK) Growth System The RCK model consists of a coupled system of two ODEs that jointly determine consumption and capital accumulation paths.$$\dot{k}(t) = f(k(t)) - c(t) - (n + g + \delta)k(t)$$$$\dot{c}(t) = \frac{c(t)}{\theta} [f'(k(t)) - \delta - \rho - \theta g]$$Variables: $k(t)$ is the capital per effective worker, $c(t)$ is the consumption per effective worker, $\rho$ is the discount rate, $\theta$ is the coefficient of relative risk aversion, and $g$ is the technological growth rate.

The RCK model is a system of two coupled ordinary differential equations that endogenizes the savings rate through representative household optimization (Wikipedia, 2026). The continuous dynamic system is defined as:  $$\dot{k}(t) = f(k(t)) - c(t) - (n + g + \delta)k(t)$$$$\dot{c}(t) = \frac{c(t)}{\theta} [f'(k(t)) - \delta - \rho - \theta g]$$Here, $k(t)$ is the capital per effective worker, $c(t)$ is the consumption per effective worker, $\alpha$ is the capital share of output, $\rho$ is the rate of time preference, $\theta$ is the inverse of the intertemporal elasticity of substitution, $\delta$ is the depreciation rate, $n$ is the population growth rate, and $g$ is the technological growth rate (Haase, 2025).1. Parameter and Initial State SpaceBecause the RCK model exhibits saddle-path stability, trajectories that do not converge will diverge or lead to negative values (MathWorks, 2026). To ensure stable numerical solutions, the initial states $k(0)$ and $c(0)$ are sampled within the neighborhood of the steady state $(k^*, c^*)$.  $$\text{Steady State Capital: } k^* = \left( \frac{\alpha}{\delta + \rho + \theta g} \right)^{\frac{1}{1-\alpha}}$$$$\text{Steady State Consumption: } c^* = (k^*)^{\alpha} - (n + g + \delta)k^*$$The bounds used for generating the random parameters are as follows:Capital elasticity ($\alpha$): $[0.3, 0.4]$Depreciation rate ($\delta$): $[0.03, 0.08]$Time preference rate ($\rho$): $[0.01, 0.05]$Risk aversion parameter ($\theta$): $[1.5, 3.0]$Population growth rate ($n$): $[0.01, 0.03]$Technological growth rate ($g$): $[0.01, 0.03]$Initial conditions $(k_0, c_0)$  $[0.8k^*, 1.2k^*]$ and $[0.8c^*, 1.2c^*]$.

In [16]:
import numpy as np
from scipy.integrate import solve_ivp

def rck_ode(t, z, alpha, delta, rho, theta, n, g):
    """
    Defines the coupled RCK ODE system.
    z[0] = k(t), z[1] = c(t)
    """
    k, c = z
    f_k = k ** alpha
    f_prime = alpha * (k ** (alpha - 1))
    
    dk_dt = f_k - c - (n + g + delta) * k
    dc_dt = (c / theta) * (f_prime - delta - rho - theta * g)
    
    return [dk_dt, dc_dt]

def generate_rck_dataset(num_samples=500, T=20, num_steps=100):
    """
    Generates training points for the RCK Growth Model.
    
    Args:
        num_samples (int): Number of different parameter combinations (M).
        T (float): Time horizon.
        num_steps (int): Number of time points to evaluate.
        
    Returns:
        X_inputs (np.ndarray): Shape (M, 8) - [alpha, delta, rho, theta, n, g, k(0), c(0)]
        Y_k (np.ndarray): Shape (M, num_steps) - k(t)
        Y_c (np.ndarray): Shape (M, num_steps) - c(t)
    """
    t_span = (0.0, T)
    t_eval = np.linspace(0.0, T, num_steps)
    
    X_inputs = []
    Y_k = []
    Y_c = []
    
    np.random.seed(42)
    
    while len(X_inputs) < num_samples:
        # Sample parameters
        alpha = np.random.uniform(0.3, 0.4)
        delta = np.random.uniform(0.03, 0.08)
        rho = np.random.uniform(0.01, 0.05)
        theta = np.random.uniform(1.5, 3.0)
        n = np.random.uniform(0.01, 0.03)
        g = np.random.uniform(0.01, 0.03)
        
        # Calculate steady state
        try:
            k_star = ((alpha) / (delta + rho + theta * g)) ** (1 / (1 - alpha))
            c_star = (k_star ** alpha) - (n + g + delta) * k_star
            
            if k_star <= 0 or c_star <= 0:
                continue
        except (ZeroDivisionError, ValueError):
            continue
            
        # Sample initial values near the steady state to avoid divergence
        k0 = np.random.uniform(0.8 * k_star, 1.2 * k_star)
        c0 = np.random.uniform(0.8 * c_star, 1.2 * c_star)
        
        sol = solve_ivp(
            rck_ode, 
            t_span, 
            [k0, c0], 
            args=(alpha, delta, rho, theta, n, g), 
            t_eval=t_eval,
            method='RK45'
        )
        
        if sol.success and sol.y.shape[1] == num_steps:
            # Check for non-divergent outputs
            if np.max(sol.y) < 1e4 and np.min(sol.y) > 0:
                X_inputs.append([alpha, delta, rho, theta, n, g, k0, c0])
                Y_k.append(sol.y[0])
                Y_c.append(sol.y[1])
                
    return np.array(X_inputs), np.array(Y_k), np.array(Y_c)



In [17]:

M = 100
X, Y_k, Y_c = generate_rck_dataset(num_samples=M)
    
print(f"Inputs Shape: {X.shape} (Features: alpha, delta, rho, theta, n, g, k0, c0)")
print(f"k(t) Trajectories Shape: {Y_k.shape}")
print(f"c(t) Trajectories Shape: {Y_c.shape}")

C:\Users\antek\AppData\Local\Temp\ipykernel_5224\1218264237.py:10: RuntimeWarning: invalid value encountered in scalar power
  f_k = k ** alpha
C:\Users\antek\AppData\Local\Temp\ipykernel_5224\1218264237.py:11: RuntimeWarning: invalid value encountered in scalar power
  f_prime = alpha * (k ** (alpha - 1))


Inputs Shape: (100, 8) (Features: alpha, delta, rho, theta, n, g, k0, c0)
k(t) Trajectories Shape: (100, 100)
c(t) Trajectories Shape: (100, 100)


# Black Scholes equation

Treating the Black-Scholes model as a Partial Differential Equation (PDE) shifts the focus from finding a single option price at $t=0$ to solving for the entire option price surface over the asset price $S$ and time $t$.When applying Neural Operators or standard PINNs (Physics-Informed Neural Networks) to the Black-Scholes PDE, you treat the model as a function mapping between continuous spaces.1. The Black-Scholes PDE FormulationThe Black-Scholes PDE is a second-order parabolic partial differential equation defined as:$$\frac{\partial V(S,t)}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V(S,t)}{\partial S^2} + rS \frac{\partial V(S,t)}{\partial S} - r V(S,t) = 0$$Subject to the terminal condition (for a European Call Option):$$V(S, T) = \max(S - K, 0)$$And boundary conditions:$V(0, t) = 0$$V(S, t) \approx S$ as $S \to \infty$

In [2]:
import numpy as np

def generate_bs_pde_grid(S_max=200, K=100, T=1.0, r=0.05, sigma=0.2, M=50, N=50):
    """
    Generates a 2D grid of option prices using the explicit finite difference method.
    
    Args:
        S_max (float): Maximum asset price.
        K (float): Strike price.
        T (float): Maturity.
        r (float): Risk-free interest rate.
        sigma (float): Volatility.
        M (int): Number of asset price steps.
        N (int): Number of time steps.
        
    Returns:
        S (np.ndarray): Asset price grid (M+1).
        t (np.ndarray): Time grid (N+1).
        V (np.ndarray): Option price surface of shape (N+1, M+1).
    """
    dt = T / N
    dS = S_max / M
    
    S = np.linspace(0, S_max, M + 1)
    t = np.linspace(0, T, N + 1)
    V = np.zeros((N + 1, M + 1))
    
    # Terminal condition at t = T
    V[N, :] = np.maximum(S - K, 0)
    
    # Boundary conditions at S = 0 and S = S_max
    V[:, 0] = 0.0
    V[:, M] = S_max - K * np.exp(-r * (T - t[::-1])) # Approximate
    
    # Iterate backward in time (Explicit Euler Scheme)
    for j in range(N, 0, -1):
        for i in range(1, M):
            delta = (V[j, i+1] - V[j, i-1]) / (2 * dS)
            gamma = (V[j, i+1] - 2 * V[j, i] + V[j, i-1]) / (dS**2)
            
            # Rearranging PDE to advance backward in time: dV/dt = ...
            # Note: Explicit Euler for BS PDE uses the negative time step
            dV_dt = 0.5 * (sigma**2) * (S[i]**2) * gamma + r * S[i] * delta - r * V[j, i]
            V[j-1, i] = V[j, i] - dV_dt * dt
            
    return S, t, V





In [3]:
S_grid, t_grid, V_surface = generate_bs_pde_grid(M=20, N=20)

print(f"Asset Price Array Shape: {S_grid.shape}")
print(f"Time Array Shape: {t_grid.shape}")
print(f"Resulting V-Surface Shape: {V_surface.shape}")
print(f"Option Price at t=0, S=100: {V_surface[0, 10]:.4f}")

Asset Price Array Shape: (21,)
Time Array Shape: (21,)
Resulting V-Surface Shape: (21, 21)
Option Price at t=0, S=100: -629.6026
